# 05 · City comparisons
Level, spread and timing for selected large cities, a common-window ranking, and a comparison of the two PM2.5 sources where they overlap.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30, "display.width", 160)

from src.data import load
from src.analysis import descriptive, trends, seasonality, correlations, statistical_tests as st, data_quality as dq, sensitivity
from src.visualization import charts, maps
from src.data.transform import independent_units

wb = descriptive.real_only(load.load_processed("wb_daily"))
cities = load.load_processed("wb_cities")

In [2]:
big = cities.sort_values('population', ascending=False)['city_label'].astype(str).head(8).tolist()
d = wb[wb['city_label'].astype(str).isin(big)].assign(city_label=lambda x: x['city_label'].astype(str))
big

['Lahore',
 'Peshawar',
 'Faisalabad',
 'Rawalpindi',
 'Gujranwala',
 'Multan',
 'Hyderabad',
 'Quetta']

In [3]:
stats = descriptive.summary_stats(d, 'pm25', 'city_label')
stats['city_label'] = stats['city_label'].astype(str)
stats.round(1)

,city_label,n,missing_pct,mean,std,min,p5,p25,p50,p75,p95,max,median
0,Faisalabad,2797.0,0.0,91.6,34.9,28.3,40.8,65.8,87.9,112.2,153.8,235.1,87.9
1,Gujranwala,2797.0,0.0,95.0,36.2,34.3,45.6,67.6,90.4,114.7,166.3,247.4,90.4
2,Hyderabad,2797.0,0.0,104.7,48.8,28.4,43.7,67.4,95.3,127.7,206.6,286.2,95.3
3,Lahore,2797.0,0.0,91.7,34.8,30.0,40.8,65.9,89.1,113.2,156.8,221.0,89.1
4,Multan,2797.0,0.0,125.2,66.6,17.0,41.4,74.3,111.4,161.2,258.7,377.0,111.4
5,Peshawar,2797.0,0.0,82.2,31.0,30.4,41.1,57.8,77.6,100.6,141.7,204.0,77.6
6,Quetta,2797.0,0.0,63.1,28.7,16.6,25.1,41.1,58.3,79.2,120.4,186.9,58.3
7,Rawalpindi,2797.0,0.0,46.3,17.6,17.9,22.6,32.9,44.6,55.8,80.8,119.0,44.6


In [4]:
charts.city_interval_dots(stats)

In [5]:
charts.box_violin(d, kind='violin')

In [6]:
charts.ecdf_chart(d)

In [7]:
charts.percentile_chart(descriptive.percentile_table(d))

In [8]:
m = trends.monthly_series(d, group_cols=('city_label',))
charts.multi_city_lines(m, 'period', 'mean', title='Monthly mean PM2.5')

In [9]:
charts.yoy_heatmap(trends.year_over_year(d, group_col='city_label'))

## Common-window ranking with artifact flags

In [10]:
descriptive.comparable_period_ranking(wb).merge(cities[['city_id','artifact_reason']], on='city_id').head(20)

,city_id,city_label,median,n,coverage,rank,artifact_reason
0,wb:1172128,Larkana,126.116776,2797,1.0,1,abrupt level shift
1,wb:1167460,Rahim Yar Khan,114.886493,2797,1.0,2,fixed annual mean + abrupt level shift
2,wb:1169825,Multan,111.446820,2797,1.0,3,fixed annual mean + abrupt level shift
3,wb:1176889,Hingorja,111.307595,2797,1.0,4,abrupt level shift
4,wb:1412008,Setharja Old,111.307595,2797,1.0,4,abrupt level shift
5,wb:1179061,Gambat,110.584548,2797,1.0,6,abrupt level shift
6,wb:1164408,Sukkur,110.097882,2797,1.0,7,fixed annual mean + abrupt level shift
7,wb:1166827,Rohri,110.097882,2797,1.0,7,fixed annual mean + abrupt level shift
8,wb:1169187,Naudero,109.083842,2797,1.0,9,abrupt level shift
9,wb:1178456,Ghotki,108.508248,2797,1.0,10,abrupt level shift


## Source-to-source comparison (Kaggle/HF vs World Bank, same city and date)

In [11]:
ov = load.load_processed('source_overlap')
dq.source_comparison(ov).round(2)

,city,wb_match,distance_km,n_days,median_hf,median_wb,mean_bias_hf_minus_wb,mae,ratio_of_medians,pearson_r,r_ci_low_adj,r_ci_high_adj,n_eff
0,Faisalabad,Faisalabad,5.79,88,154.25,128.34,26.88,58.24,1.20,-0.01,-0.32,0.30,39.89
1,Islamabad,Islamabad,4.15,88,57.13,61.48,-3.99,19.70,0.93,0.13,-0.17,0.41,44.10
2,Karachi,Karachi University Employees Co-operative Hous...,16.47,88,38.40,57.17,-20.52,21.70,0.67,0.40,0.14,0.61,51.24
3,Lahore,Lahore,1.14,88,125.34,113.95,9.40,57.90,1.10,-0.15,-0.50,0.25,27.32
4,Multan,Multan,6.26,88,133.46,93.31,40.84,53.88,1.43,0.11,-0.23,0.43,34.54
5,Peshawar,Peshawar,5.00,88,51.85,97.89,-51.34,56.04,0.53,-0.10,-0.43,0.25,32.41
6,Quetta,Quetta,2.58,88,18.71,42.64,-24.59,25.84,0.44,-0.28,-0.55,0.05,37.93
7,Rahim Yar Khan,Rahim Yar Khan,0.81,88,68.97,117.49,-44.77,51.71,0.59,0.18,-0.11,0.44,47.61
8,Rawalpindi,Rawalpindi,4.59,88,57.13,61.94,-8.98,22.83,0.92,0.19,-0.16,0.51,32.67
9,Sialkot,Sialkot,0.82,88,107.60,96.09,11.45,54.00,1.12,-0.21,-0.55,0.19,27.03


In [12]:
charts.source_comparison_scatter(ov)